# 23 · Gray-Box AIA72 Rolling Operational Replay

Phase G performs a prospective-style chronological replay of the frozen 72 h multimodal Gray-Box framework over Cycle 25 and the supplementary 2026 period.

**Key rule:** each forecast decision is made from the frozen model outputs and frozen v3 trust policy available at issue time. Labels are treated as unavailable until the 72 h forecast horizon plus a fixed 24 h reporting delay has elapsed. Rolling monitoring therefore uses only outcomes that would already have matured. No model fitting, recalibration, threshold tuning, policy reselection, or redesign occurs during the replay.

Two scenarios are reported: (1) nominal frozen-input replay and (2) replay with mapped real AIA acquisition incidents forcing the AIA branch unavailable and therefore routing to SHARP fallback when possible. The second scenario is an engineering robustness replay, not a claim about verified historical live-service delivery.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json
import numpy as np
import pandas as pd
from sklearn.metrics import brier_score_loss, average_precision_score, roc_auc_score

ROOT=Path('/home/abmoses2000')
MASTER=ROOT/'graybox_aia72_master_predictions_v2_20261005/graybox_aia72_master_predictions.csv.gz'
MASTER_SHA='17300093b88b7c61ecadf5eb28acba0d6d023b340d7740807852dd4352b22b46'
SHARP=ROOT/'sharp72_predictions_frozen_20261002.csv.gz'
SHARP_SHA='7368c002a978321cdc67221564f3b7901687badc7aba28ab9df615179d1e017b'
MOND=ROOT/'graybox_aia72_conformal_mondrian_v2_20261005/mondrian_thresholds.csv'
CUTS=ROOT/'graybox_aia72_policy_freeze_v1_20261005/candidate_cutoffs.csv'
ALARM=ROOT/'graybox_aia72_alarm_and_primary_policy_freeze_v1_20261005/alarm_thresholds.csv'
INC_MAP=ROOT/'graybox_aia72_real_outage_replay_v1_20261006/incident_to_graybox_mapping.csv'
OUT=ROOT/'graybox_aia72_rolling_operational_replay_v1_20261006'

if OUT.exists():
    raise SystemExit(f'Refusing to overwrite existing output: {OUT}')
OUT.mkdir(parents=True)

def sha256(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as f:
        for c in iter(lambda:f.read(4*1024*1024),b''): h.update(c)
    return h.hexdigest()

def require(c,m):
    if not c: raise ValueError(m)

for p in [MASTER,SHARP,MOND,CUTS,ALARM]:
    require(p.is_file(),f'Missing frozen input: {p}')
require(sha256(MASTER)==MASTER_SHA,'Master SHA changed')
require(sha256(SHARP)==SHARP_SHA,'SHARP SHA changed')

HORIZON_H=72
REPORT_DELAY_H=24
ISSUE_CADENCE_MIN=96
PRIMARY_PCT=.90
SEEDS=[17,29,43]
ROLES=['retrospective_cycle25','supplementary_2026']

def entropy_binary(p):
    q=np.clip(np.asarray(p,float),1e-12,1-1e-12)
    return -(q*np.log(q)+(1-q)*np.log(1-q))

def mondrian_state(p,q0,q1):
    p=np.asarray(p,float); i0=p<=q0; i1=(1-p)<=q1
    out=np.empty(len(p),dtype=object)
    out[(~i0)&(~i1)]='empty'
    out[i0&(~i1)]='singleton_no_flare'
    out[(~i0)&i1]='singleton_flare'
    out[i0&i1]='doubleton'
    return out

def binary_metrics(y,pred):
    y=np.asarray(y,int); pred=np.asarray(pred,int)
    tp=int(((y==1)&(pred==1)).sum()); fp=int(((y==0)&(pred==1)).sum())
    fn=int(((y==1)&(pred==0)).sum()); tn=int(((y==0)&(pred==0)).sum())
    rec=tp/(tp+fn) if tp+fn else np.nan
    spec=tn/(tn+fp) if tn+fp else np.nan
    prec=tp/(tp+fp) if tp+fp else np.nan
    return {'tp':tp,'fp':fp,'fn':fn,'tn':tn,'recall':rec,'specificity':spec,
            'precision':prec,'tss':rec+spec-1 if np.isfinite(rec) and np.isfinite(spec) else np.nan}

master=pd.read_csv(MASTER,low_memory=False)
sharp=pd.read_csv(SHARP,low_memory=False)
a=master[master.role.astype(str).isin(ROLES)].copy()
s=sharp[sharp.role.astype(str).isin(ROLES)].copy()
a.forecast_case_id=a.forecast_case_id.astype(str); s.forecast_case_id=s.forecast_case_id.astype(str)
require(set(a.forecast_case_id)==set(s.forecast_case_id),'Evaluation ID mismatch')
m=a.merge(s,on='forecast_case_id',suffixes=('_aia','_sharp'),validate='one_to_one')

# issue time hierarchy
issue=pd.Series(pd.NaT,index=m.index,dtype='datetime64[ns, UTC]')
for c in [x for x in ['issue_utc_aia','issue_utc_sharp','issue_utc'] if x in m.columns]:
    z=pd.to_datetime(m[c],utc=True,errors='coerce'); take=issue.isna()&z.notna(); issue.loc[take]=z.loc[take]
id_time=m.forecast_case_id.str.extract(r':(\d{8}_\d{4})_',expand=False)
z=pd.to_datetime(id_time,format='%Y%m%d_%H%M',utc=True,errors='coerce'); take=issue.isna()&z.notna(); issue.loc[take]=z.loc[take]
require(issue.notna().all(),'Unresolved issue time')
m['issue_utc']=issue
m['outcome_available_utc']=m.issue_utc+pd.Timedelta(hours=HORIZON_H+REPORT_DELAY_H)

la='label_aia' if 'label_aia' in m.columns else 'label'
ls='label_sharp' if 'label_sharp' in m.columns else la
m['label']=pd.to_numeric(m[la],errors='raise').astype(int)
require(np.array_equal(m.label.to_numpy(),pd.to_numeric(m[ls],errors='raise').astype(int).to_numpy()),'Label mismatch')
role_col='role_aia' if 'role_aia' in m.columns else ('role_sharp' if 'role_sharp' in m.columns else 'role')
m['role']=m[role_col].astype(str)

# frozen branch outputs
m['sharp_probability']=pd.to_numeric(m.probability_gru_mean,errors='coerce')
m['aia_probability']=pd.to_numeric(m.probability,errors='coerce')
m['fusion_probability']=.5*(m.sharp_probability+m.aia_probability)

sharp_seed=m[[f'probability_seed_{z}_sharp' if f'probability_seed_{z}_sharp' in m.columns else f'probability_seed_{z}' for z in SEEDS]].astype(float).to_numpy()
aia_cols=[]
for z in SEEDS:
    if f'probability_seed_{z}_aia' in m.columns: aia_cols.append(f'probability_seed_{z}_aia')
    elif f'probability_seed_{z}' in a.columns: aia_cols.append(f'probability_seed_{z}')
    else: raise ValueError(f'Missing AIA seed {z}')
aia_seed=m[aia_cols].astype(float).to_numpy()
fusion_seed=.5*(sharp_seed+aia_seed)

m['sharp_seed_std']=np.nanstd(sharp_seed,axis=1,ddof=1)
m['aia_seed_std']=np.nanstd(aia_seed,axis=1,ddof=1)
m['fusion_seed_std']=np.nanstd(fusion_seed,axis=1,ddof=1)
m['sharp_aia_probability_gap']=np.abs(m.sharp_probability-m.aia_probability)
m['fusion_entropy']=entropy_binary(m.fusion_probability)

aia_status='input_status_aia' if 'input_status_aia' in m.columns else ('input_status' if 'input_status' in m.columns else None)
m['aia_technical_ok']=np.isfinite(m.aia_probability)&np.isfinite(aia_seed).all(axis=1)
if aia_status is not None:
    m['aia_technical_ok'] &= m[aia_status].astype(str).eq('ok')
m['sharp_technical_ok']=np.isfinite(m.sharp_probability)&np.isfinite(sharp_seed).all(axis=1)

mond=pd.read_csv(MOND)
q={}
for branch in ['sharp','aia','sharp_aia']:
    h=mond[(mond.branch==branch)&np.isclose(mond.alpha.astype(float),.10)]
    require(len(h)==1,f'Missing Mondrian thresholds {branch}')
    q[branch]=(float(h.iloc[0].qhat_class0),float(h.iloc[0].qhat_class1))
m['fusion_mondrian_state']=mondrian_state(m.fusion_probability,*q['sharp_aia'])
m['aia_mondrian_state']=mondrian_state(m.aia_probability,*q['aia'])
m['sharp_mondrian_state']=mondrian_state(m.sharp_probability,*q['sharp'])

cuts=pd.read_csv(CUTS)
def cutoff(metric,pct=.90):
    h=cuts[(cuts.metric==metric)&np.isclose(cuts.percentile.astype(float),pct)]
    require(len(h)==1,f'Missing cutoff {metric}')
    return float(h.iloc[0].cutoff)
FS=cutoff('fusion_seed_std'); GAP=cutoff('sharp_aia_probability_gap'); ENT=cutoff('fusion_entropy')

alarm=pd.read_csv(ALARM)
ath={r.branch:float(r.threshold) for r in alarm.itertuples()}

# Real acquisition incidents mapped in Phase E1.
real_incident_ids=set()
if INC_MAP.is_file():
    im=pd.read_csv(INC_MAP,low_memory=False)
    if 'sample_id' in im.columns:
        # forecast_case_id embeds source sample IDs in this build, but use row_index only if exact ID absent.
        sid=set(im.sample_id.astype(str))
        src_col='source_sample_id_aia' if 'source_sample_id_aia' in m.columns else ('source_sample_id' if 'source_sample_id' in m.columns else None)
        if src_col is not None:
            real_incident_ids=set(m.loc[m[src_col].astype(str).isin(sid),'forecast_case_id'].astype(str))
        else:
            # Conservative fallback: Phase E1 mapping row_index was against full master order.
            full=master.reset_index().rename(columns={'index':'row_index'})
            ids=set(full.loc[full.row_index.isin(pd.to_numeric(im.row_index,errors='coerce').dropna().astype(int)),'forecast_case_id'].astype(str))
            real_incident_ids=ids

def apply_policy(df,scenario):
    aia_ok=df.aia_technical_ok.to_numpy(bool).copy()
    if scenario=='real_aia_incident_replay':
        outage=df.forecast_case_id.astype(str).isin(real_incident_ids).to_numpy()
        aia_ok &= ~outage
    else:
        outage=np.zeros(len(df),dtype=bool)
    sharp_ok=df.sharp_technical_ok.to_numpy(bool)
    fusion_ok=aia_ok & sharp_ok
    fusion_single=df.fusion_mondrian_state.isin(['singleton_no_flare','singleton_flare']).to_numpy()
    normal=(fusion_ok & fusion_single &
            (df.fusion_seed_std.to_numpy()<=FS) &
            (df.sharp_aia_probability_gap.to_numpy()<=GAP) &
            (df.fusion_entropy.to_numpy()<=ENT))
    degraded=(~normal)&sharp_ok
    abstain=~(normal|degraded)
    state=np.where(normal,'NORMAL',np.where(degraded,'DEGRADED','ABSTAIN'))
    source=np.where(normal,'fusion',np.where(degraded,'sharp','none'))
    prob=np.where(normal,df.fusion_probability,np.where(degraded,df.sharp_probability,np.nan))
    pred=np.where(normal,prob>=ath['sharp_aia'],np.where(degraded,prob>=ath['sharp'],False))
    out=df[['forecast_case_id','issue_utc','outcome_available_utc','role','label']].copy()
    out['scenario']=scenario; out['state']=state; out['forecast_source']=source
    out['issued_probability']=prob; out['issued_alarm']=pred.astype(int)
    out['real_aia_incident_applied']=outage
    out['fusion_entropy']=df.fusion_entropy.to_numpy()
    out['fusion_seed_std']=df.fusion_seed_std.to_numpy()
    out['sharp_aia_probability_gap']=df.sharp_aia_probability_gap.to_numpy()
    out['fusion_mondrian_state']=df.fusion_mondrian_state.to_numpy()
    out['aia_mondrian_state']=df.aia_mondrian_state.to_numpy()
    return out.sort_values(['issue_utc','forecast_case_id']).reset_index(drop=True)

logs=pd.concat([apply_policy(m,'nominal_frozen_inputs'),
                apply_policy(m,'real_aia_incident_replay')],ignore_index=True)
logs.to_csv(OUT/'operational_replay_log.csv.gz',index=False,compression={'method':'gzip','mtime':0})

# Cadence audit: within forecast case source/AR where a usable HARP is embedded in ID.
# Do not force exact 96-min equality; summarize observed issue spacing.
tmp=m[['forecast_case_id','issue_utc']].copy()
tmp['harp']=tmp.forecast_case_id.str.extract(r'HARP(\d+)',expand=False)
tmp=tmp[tmp.harp.notna()].sort_values(['harp','issue_utc'])
tmp['delta_min']=tmp.groupby('harp').issue_utc.diff().dt.total_seconds()/60
cad=tmp.delta_min.dropna()
cadence={'declared_cadence_minutes':ISSUE_CADENCE_MIN,'observed_interissue_count':int(len(cad)),
         'median_within_harp_delta_minutes':float(cad.median()) if len(cad) else None,
         'fraction_exact_96_minutes':float(np.isclose(cad,96).mean()) if len(cad) else None,
         'fraction_within_90_102_minutes':float(cad.between(90,102).mean()) if len(cad) else None}
(OUT/'cadence_audit.json').write_text(json.dumps(cadence,indent=2)+'\n')

# End-of-replay performance (labels may be used here because this is retrospective scoring,
# never policy construction).
rows=[]
for (scenario,role),g in logs.groupby(['scenario','role'],sort=False):
    issued=g.state.ne('ABSTAIN')
    y=g.loc[issued,'label'].to_numpy(int); pred=g.loc[issued,'issued_alarm'].to_numpy(int)
    r={'scenario':scenario,'role':role,'cases':len(g),'issued_cases':int(issued.sum()),
       'availability_rate':float(issued.mean()),
       'normal_rate':float(g.state.eq('NORMAL').mean()),
       'degraded_rate':float(g.state.eq('DEGRADED').mean()),
       'abstain_rate':float(g.state.eq('ABSTAIN').mean()),
       'positive_retention_rate':float(issued[g.label.eq(1)].mean()) if g.label.eq(1).any() else np.nan,
       'real_aia_incident_cases':int(g.real_aia_incident_applied.sum())}
    if len(y):
        r.update(binary_metrics(y,pred))
        p=g.loc[issued,'issued_probability'].to_numpy(float)
        r['brier']=float(brier_score_loss(y,p))
        r['average_precision']=float(average_precision_score(y,p)) if len(np.unique(y))==2 else np.nan
        r['roc_auc']=float(roc_auc_score(y,p)) if len(np.unique(y))==2 else np.nan
    rows.append(r)
summary=pd.DataFrame(rows)
summary.to_csv(OUT/'final_replay_summary.csv',index=False)

# Monthly issue/state mix.
mon=logs.copy(); mon['issue_month']=mon.issue_utc.dt.to_period('M').astype(str)
monthly=mon.groupby(['scenario','issue_month','state']).size().unstack(fill_value=0).reset_index()
for c in ['NORMAL','DEGRADED','ABSTAIN']:
    if c not in monthly: monthly[c]=0
monthly['total']=monthly[['NORMAL','DEGRADED','ABSTAIN']].sum(axis=1)
for c in ['NORMAL','DEGRADED','ABSTAIN']:
    monthly[c.lower()+'_rate']=monthly[c]/monthly.total
monthly.to_csv(OUT/'monthly_state_mix.csv',index=False)

# Past-only rolling monitor at month-end.
# At monitor time t, include ONLY cases with outcome_available_utc <= t.
monitor=[]
for scenario,g in logs.groupby('scenario'):
    g=g.sort_values('issue_utc')
    start=g.issue_utc.min().normalize()
    end=g.issue_utc.max().normalize()+pd.offsets.MonthEnd(0)
    for t in pd.date_range(start,end,freq='ME',tz='UTC'):
        matured=g[g.outcome_available_utc<=t]
        issued=matured[matured.state.ne('ABSTAIN')]
        if len(matured)==0:
            continue
        r={'scenario':scenario,'monitor_time_utc':t.isoformat(),
           'matured_cases':len(matured),'issued_matured_cases':len(issued),
           'latest_matured_issue_utc':matured.issue_utc.max(),
           'normal_rate_to_date':float(matured.state.eq('NORMAL').mean()),
           'degraded_rate_to_date':float(matured.state.eq('DEGRADED').mean()),
           'abstain_rate_to_date':float(matured.state.eq('ABSTAIN').mean())}
        if len(issued) and issued.label.nunique()==2:
            met=binary_metrics(issued.label,issued.issued_alarm)
            r.update({f'expanding_{k}':v for k,v in met.items()})
            r['expanding_brier']=float(brier_score_loss(issued.label,issued.issued_probability))
        # trailing 180d, still only matured outcomes
        lo=t-pd.Timedelta(days=180)
        tr=matured[(matured.issue_utc>=lo)&matured.state.ne('ABSTAIN')]
        r['trailing180_issued_cases']=len(tr)
        r['trailing180_positives']=int(tr.label.sum()) if len(tr) else 0
        if len(tr) and tr.label.nunique()==2:
            met=binary_metrics(tr.label,tr.issued_alarm)
            r.update({f'trailing180_{k}':v for k,v in met.items()})
            r['trailing180_brier']=float(brier_score_loss(tr.label,tr.issued_probability))
        monitor.append(r)
pd.DataFrame(monitor).to_csv(OUT/'past_only_monthly_monitor.csv',index=False)

# UQ transfer monitor, outcome-matured only. These are diagnostics and do not alter routing.
uq=[]
nom=logs[logs.scenario.eq('nominal_frozen_inputs')].copy()
for t in pd.date_range(nom.issue_utc.min().normalize(),
                       nom.issue_utc.max().normalize()+pd.offsets.MonthEnd(0),
                       freq='ME',tz='UTC'):
    z=nom[nom.outcome_available_utc<=t]
    if len(z)==0: continue
    flare=z.label.eq(1)
    # fusion true-class containment from fixed Mondrian state encoding:
    fstate=z.fusion_mondrian_state.astype(str)
    contains=np.where(z.label.to_numpy()==1,
                      fstate.isin(['singleton_flare','doubleton']).to_numpy(),
                      fstate.isin(['singleton_no_flare','doubleton']).to_numpy())
    uq.append({'monitor_time_utc':t.isoformat(),'matured_cases':len(z),'matured_positives':int(flare.sum()),
               'fusion_flare_conformal_coverage_to_date':float(np.mean(contains[flare.to_numpy()])) if flare.any() else np.nan,
               'fusion_singleton_rate_to_date':float(fstate.isin(['singleton_flare','singleton_no_flare']).mean()),
               'median_fusion_entropy_to_date':float(z.fusion_entropy.median()),
               'median_fusion_seed_std_to_date':float(z.fusion_seed_std.median()),
               'median_sharp_aia_gap_to_date':float(z.sharp_aia_probability_gap.median())})
pd.DataFrame(uq).to_csv(OUT/'past_only_uq_monitor.csv',index=False)

protocol={
    'status':'PROSPECTIVE_STYLE_ROLLING_OPERATIONAL_REPLAY_COMPLETE',
    'created_utc':datetime.now(timezone.utc).isoformat(),
    'horizon_hours':HORIZON_H,
    'declared_issue_cadence_minutes':ISSUE_CADENCE_MIN,
    'outcome_reporting_delay_hours':REPORT_DELAY_H,
    'outcome_maturity_lag_hours':HORIZON_H+REPORT_DELAY_H,
    'primary_policy':'fallback_v3_q90',
    'new_model_fitting':False,'new_recalibration':False,'new_threshold_tuning':False,
    'new_policy_selection':False,'new_policy_redesign':False,
    'rolling_monitor_affects_decisions':False,
    'scenarios':['nominal_frozen_inputs','real_aia_incident_replay'],
    'real_outage_semantics':'mapped detected AIA acquisition/recovery incidents; engineering robustness replay, not verified historical live-service delivery',
    'aia_only_fallback':'not enabled; Phase F did not support automatic AIA-only fallback when SHARP is unavailable',
    'past_only_rule':'At monitoring time t, only labels with issue_time + 72 h horizon + 24 h reporting delay <= t are visible.',
    'evidence_boundary':{'retrospective_cycle25':'prospective-style post-hoc chronological replay','supplementary_2026':'already-spent locked evidence; no redesign'},
    'source_sha256':{'master_predictions':MASTER_SHA,'sharp_predictions':SHARP_SHA,
                     'mondrian_thresholds':sha256(MOND),'candidate_cutoffs':sha256(CUTS),'alarm_thresholds':sha256(ALARM)},
    'outputs':['operational_replay_log.csv.gz','cadence_audit.json','final_replay_summary.csv',
               'monthly_state_mix.csv','past_only_monthly_monitor.csv','past_only_uq_monitor.csv']
}
if INC_MAP.is_file(): protocol['source_sha256']['incident_to_graybox_mapping']=sha256(INC_MAP)
(OUT/'protocol.json').write_text(json.dumps(protocol,indent=2)+'\n')

print('===== ROLLING OPERATIONAL REPLAY COMPLETE =====')
print('\nFinal summary:')
print(summary.round(6).to_string(index=False))
print('\nCadence audit:')
print(json.dumps(cadence,indent=2))
print('\nLast five past-only monitor rows:')
pm=pd.read_csv(OUT/'past_only_monthly_monitor.csv')
print(pm.tail().round(6).to_string(index=False))
print('\nLast five UQ monitor rows:')
pu=pd.read_csv(OUT/'past_only_uq_monitor.csv')
print(pu.tail().round(6).to_string(index=False))
print('\nOutputs:',OUT)


## Interpretation guardrails

- This is a **prospective-style replay**, not a truly prospective deployment trial.
- Frozen model probabilities and trust thresholds are replayed in chronological order; the rolling monitor never feeds back into the policy.
- Outcomes become visible only after 96 h (72 h horizon + 24 h reporting delay), preventing future-label leakage in monitoring.
- Real AIA acquisition incidents are replayed as a robustness scenario only. They do not establish verified live-service outages.
- Because Phase F did not support AIA-only fallback, SHARP unavailability remains an ABSTAIN condition in the current 72 h architecture.
